In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"adityakumar9876","key":"88247f7e2a92f48bb7fef5d1b8c2c028"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d chethuhn/network-intrusion-dataset
!unzip -q network-intrusion-dataset.zip -d cicids2017

Dataset URL: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
License(s): CC0-1.0
100% 230M/230M [00:08<00:00, 29.0MB/s]



In [4]:
import pandas as pd
import numpy as np
import glob
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Define the exact features our live Scapy engine can capture
# Added 'Label' here so we can filter for BENIGN traffic during loading
columns_to_keep = [
    'Destination Port',
    'FIN Flag Count',
    'SYN Flag Count',
    'Total Length of Fwd Packets',
    'Label'
]

print("Loading dataset chunks...")
file_paths = glob.glob('cicids2017/*.csv')
train_samples = []

# 2. Memory-efficient data loading
for file in file_paths:
    chunk_iter = pd.read_csv(file, chunksize=100000)
    for chunk in chunk_iter:
        # Clean column names (strip hidden whitespace)
        chunk.columns = chunk.columns.str.strip()

        # Keep only the features we need
        available_cols = [c for c in columns_to_keep if c in chunk.columns]
        chunk = chunk[available_cols]

        # We only want to train the model on 'Normal' (BENIGN) traffic
        if 'Label' in chunk.columns:
            benign_traffic = chunk[chunk['Label'] == 'BENIGN']
            if not benign_traffic.empty:
                # Sample 10% to keep training fast and lightweight
                sampled_benign = benign_traffic.sample(frac=0.10, random_state=42)
                train_samples.append(sampled_benign)

if train_samples:
    df_train_lite = pd.concat(train_samples, ignore_index=True)
    print(f"Data loaded. Training shape: {df_train_lite.shape}")
else:
    print("Error: No BENIGN samples found. Check if 'Label' exists and contains 'BENIGN' strings.")

Loading dataset chunks...
Data loaded. Training shape: (227310, 5)


In [5]:
# 1. Prepare Features (Drop the Label)
X_train_lite = df_train_lite.drop(columns=['Label'])

# Handle any infinites or NaNs
X_train_lite.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train_lite.fillna(0, inplace=True)

print("Scaling data...")
scaler_lite = StandardScaler()
# Fit the scaler
X_train_scaled_lite = scaler_lite.fit_transform(X_train_lite)

print("Training Lightweight Isolation Forest...")
iso_forest_lite = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.01, # Expecting 1% of traffic to be anomalous
    random_state=42,
    n_jobs=-1
)

iso_forest_lite.fit(X_train_scaled_lite)
print("Training Complete!")

Scaling data...
Training Lightweight Isolation Forest...
Training Complete!


In [6]:
# Create the deployment payload
project_artifacts_lite = {
    'model': iso_forest_lite,
    'scaler': scaler_lite,
    'threshold': -0.11, # Our manually tuned production threshold
    'features': X_train_lite.columns.tolist() # Strict feature enforcement
}

# Export the model
joblib.dump(project_artifacts_lite, 'ids_model_lite.pkl')
print("Model exported successfully as 'ids_model_lite.pkl'")

# Quick verification test
print("\n--- Artifact Verification ---")
print("Expected Features for Live Sniffer:")
for feature in project_artifacts_lite['features']:
    print(f"- {feature}")

Model exported successfully as 'ids_model_lite.pkl'

--- Artifact Verification ---
Expected Features for Live Sniffer:
- Destination Port
- FIN Flag Count
- SYN Flag Count
- Total Length of Fwd Packets
